# Previsão de Satisfação de Clientes — Olist Brazilian E-Commerce

**Trabalho Final — Ciência de Dados**  
Universidade de Passo Fundo — UPF  

**Autores:** Murilo Moreira Mognon e Érick Landim de Paula
**Data:** 2025  

---

**Repositório GitHub:** [https://github.com/mognon-murilo/predicting-customer-satisfaction-olist.git](https://github.com/mognon-murilo/predicting-customer-satisfaction-olist.git)

## 1. Introdução

O presente trabalho tem como objetivo desenvolver um modelo de aprendizado de máquina capaz de prever a satisfação de clientes de uma plataforma de e-commerce. O problema é formulado como **classificação binária**: dado um conjunto de informações sobre um pedido (prazo de entrega, valor, frete, categoria do produto, entre outros), o modelo deve prever se o cliente ficará **satisfeito** (nota de avaliação ≥ 4) ou **insatisfeito** (nota ≤ 3).

### Relevância

Identificar clientes insatisfeitos antes mesmo de receberem o produto permite que a empresa tome ações proativas — como reembolsos automáticos, contato preventivo ou ajuste logístico — reduzindo churn e aumentando a retenção. No contexto do e-commerce brasileiro, onde a satisfação com o prazo de entrega é um dos principais fatores de fidelização, esse tipo de modelo tem alto valor estratégico.

### Hipóteses iniciais

As seguintes hipóteses nortearam a análise:

1. **Atraso na entrega** é o principal fator de insatisfação.
2. **Frete alto** em relação ao valor do pedido reduz a satisfação.
3. A **categoria do produto** influencia a expectativa e, consequentemente, a avaliação.
4. Pedidos **interestaduais** têm mais atrasos e menor satisfação.
5. **Pagamentos parcelados** estão associados a pedidos maiores e piores avaliações.

## 2. Dataset

### 2.1 Origem

O dataset utilizado é o **Olist Brazilian E-Commerce Dataset**, disponível publicamente no Kaggle ([link](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)). Contém informações reais de pedidos realizados na plataforma Olist entre 2016 e 2018, distribuídos em 9 tabelas relacionais que foram unificadas em uma única tabela analítica via joins.

**Dimensões após join:** ~99.441 linhas × 24 colunas  
**Após limpeza:** 86.811 linhas × 138 colunas (com dummies)

### 2.2 Dicionário de Variáveis

#### Tabela principal: `olist_orders_dataset`
| Variável | Tipo | Descrição |
|---|---|---|
| order_id | string | Identificador único do pedido |
| customer_id | string | Identificador do cliente |
| order_status | string | Status do pedido (delivered, canceled, etc.) |
| order_purchase_timestamp | datetime | Data/hora da compra |
| order_approved_at | datetime | Data/hora de aprovação do pagamento |
| order_delivered_carrier_date | datetime | Data de entrega ao transportador |
| order_delivered_customer_date | datetime | Data de entrega ao cliente |
| order_estimated_delivery_date | datetime | Data estimada de entrega |

#### Tabela: `olist_order_items_dataset`
| Variável | Tipo | Descrição |
|---|---|---|
| order_id | string | Identificador do pedido |
| product_id | string | Identificador do produto |
| seller_id | string | Identificador do vendedor |
| price | float | Preço unitário do item |
| freight_value | float | Valor do frete por item |

#### Tabela: `olist_order_reviews_dataset` (variável-alvo)
| Variável | Tipo | Descrição |
|---|---|---|
| review_score | int | Nota de 1 a 5 dada pelo cliente |
| review_comment_message | string | Comentário textual (não utilizado) |

#### Tabela: `olist_order_payments_dataset`
| Variável | Tipo | Descrição |
|---|---|---|
| payment_type | string | Tipo de pagamento (credit_card, boleto, etc.) |
| payment_installments | int | Número de parcelas |
| payment_value | float | Valor total pago |

#### Features criadas por engenharia
| Feature | Descrição |
|---|---|
| delivery_delay_days | Diferença em dias entre entrega real e estimada (negativo = adiantado) |
| shipping_days | Dias entre aprovação e envio ao transportador |
| estimated_delivery_days | Prazo estimado em dias no momento da compra |
| is_late | Flag binária: 1 se o pedido chegou após a data estimada |
| freight_ratio | Proporção do frete em relação ao valor total do pedido |
| price_total | Soma dos preços de todos os itens do pedido |
| freight_total | Soma do frete de todos os itens do pedido |
| n_items | Quantidade de itens no pedido |
| **target** | **1 = satisfeito (review_score ≥ 4), 0 = insatisfeito (review_score ≤ 3)** |

### 2.3 Pré-processamento

As seguintes etapas foram realizadas em sequência:

1. **Filtragem:** mantidos apenas pedidos com `order_status == 'delivered'` (únicos com review confiável)
2. **Remoção de nulos no target:** linhas sem `review_score` foram descartadas
3. **Tratamento de ausentes:** medianas para numéricas, `'unknown'` para categóricas
4. **Remoção de outliers:** método IQR com multiplicador 3.0 (conservador) nas colunas `price_total`, `freight_total`, `delivery_delay_days` e `payment_value`
5. **Encoding:** One-Hot Encoding com `drop_first=True` em `payment_type`, `customer_state`, `seller_state` e `product_category_name_english`
6. **Normalização:** StandardScaler nas 11 features numéricas
7. **Split:** 80% treino / 20% teste, estratificado pela variável-alvo

## 3. Análise Exploratória dos Dados (EDA)

A análise exploratória investigou as cinco hipóteses iniciais com gráficos e estatísticas descritivas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

matplotlib.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', font_scale=1.0)

ROOT = Path('..') 
DATA_PROC = ROOT / 'data' / 'processed'

df = pd.read_csv(DATA_PROC / 'olist_analytical.csv')
df = df[df['order_status'] == 'delivered'].dropna(subset=['review_score']).copy()

for c in ['order_purchase_timestamp','order_approved_at',
          'order_delivered_carrier_date','order_delivered_customer_date',
          'order_estimated_delivery_date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['shipping_days'] = (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.days
df['is_late'] = (df['delivery_delay_days'] > 0).astype(int)
df['freight_ratio'] = df['freight_total'] / df['price_total'].replace(0, np.nan)
df['target'] = (df['review_score'] >= 4).astype(int)
df['satisfaction'] = df['target'].map({1: 'Satisfeito', 0: 'Insatisfeito'})
df['cross_state'] = (df['customer_state'] != df['seller_state']).astype(int)

print(f'Linhas: {len(df):,} | Satisfeitos: {df["target"].mean():.1%} | Insatisfeitos: {1-df["target"].mean():.1%}')

### Hipótese 1 — Atraso na entrega impacta negativamente a satisfação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Nota média por status de atraso
late_score = df.groupby('is_late')['review_score'].mean()
axes[0].bar(['No prazo', 'Com atraso'], late_score.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
axes[0].set_title('Nota média por status de entrega')
axes[0].set_ylabel('Review score médio')
axes[0].set_ylim(0, 5)
for i, v in enumerate(late_score.values):
    axes[0].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')

# Distribuição de delay para cada classe
sat = df[df['target']==1]['delivery_delay_days'].clip(-30, 60)
ins = df[df['target']==0]['delivery_delay_days'].clip(-30, 60)
axes[1].hist(sat, bins=40, alpha=0.6, color='#2ecc71', label='Satisfeito', density=True)
axes[1].hist(ins, bins=40, alpha=0.6, color='#e74c3c', label='Insatisfeito', density=True)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1, label='Prazo estimado')
axes[1].set_title('Distribuição de dias de atraso por classe')
axes[1].set_xlabel('delivery_delay_days')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Nota média — no prazo: {late_score[0]:.2f} | com atraso: {late_score[1]:.2f}")
print(f"Taxa de insatisfação com atraso: {df[df['is_late']==1]['target'].apply(lambda x: 1-x).mean():.1%}")

**Conclusão H1:** Pedidos entregues com atraso têm nota média significativamente menor. A distribuição de `delivery_delay_days` mostra que clientes insatisfeitos concentram-se claramente no lado positivo (atraso). **Hipótese confirmada** — é_late e delivery_delay_days são as features mais importantes do modelo.

### Hipótese 2 — Frete alto em relação ao valor do pedido reduz a satisfação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot de freight_ratio por classe
data_plot = [df[df['target']==1]['freight_ratio'].dropna().clip(0,2),
             df[df['target']==0]['freight_ratio'].dropna().clip(0,2)]
bp = axes[0].boxplot(data_plot, labels=['Satisfeito', 'Insatisfeito'],
                     patch_artist=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
axes[0].set_title('Freight ratio por classe')
axes[0].set_ylabel('freight_ratio (frete / valor total)')

# Scatter: freight_ratio vs review_score
sample = df.dropna(subset=['freight_ratio']).sample(3000, random_state=42)
axes[1].scatter(sample['freight_ratio'].clip(0,2), sample['review_score'],
                alpha=0.15, color='#3498db', s=8)
axes[1].set_xlabel('freight_ratio')
axes[1].set_ylabel('review_score')
axes[1].set_title('Frete relativo vs Avaliação (amostra 3k)')

plt.tight_layout()
plt.show()

fr_sat = df[df['target']==1]['freight_ratio'].median()
fr_ins = df[df['target']==0]['freight_ratio'].median()
print(f"Mediana freight_ratio — satisfeitos: {fr_sat:.3f} | insatisfeitos: {fr_ins:.3f}")

**Conclusão H2:** Clientes insatisfeitos apresentam mediana de freight_ratio maior. **Hipótese parcialmente confirmada** — a relação existe mas não é o fator dominante.

### Hipótese 3 — Categoria do produto influencia a avaliação

In [ ]:
cat_score = (df.groupby('product_category_name_english')['review_score']
               .agg(['mean','count'])
               .query('count >= 200')
               .sort_values('mean'))

fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ['#e74c3c' if v < 3.8 else '#f1c40f' if v < 4.2 else '#2ecc71'
              for v in cat_score['mean']]
bars = ax.barh(cat_score.index[-20:], cat_score['mean'][-20:],
               color=colors_bar[-20:], edgecolor='white')
ax.axvline(df['review_score'].mean(), color='black', linestyle='--',
           linewidth=1, label=f'Média geral ({df["review_score"].mean():.2f})')
ax.set_xlabel('Nota média')
ax.set_title('Top 20 categorias por nota média (mín. 200 pedidos)')
ax.legend()
ax.set_xlim(3, 5)
plt.tight_layout()
plt.show()

**Conclusão H3:** Existe variação significativa de satisfação entre categorias. **Hipótese confirmada** — a categoria do produto foi incluída como feature via One-Hot Encoding.

### Hipótese 4 — Pedidos interestaduais têm mais atrasos e menor satisfação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cross_late = df.groupby('cross_state')['is_late'].mean()
axes[0].bar(['Mesmo estado', 'Interestadual'], cross_late.values,
            color=['#3498db', '#e67e22'], edgecolor='white', width=0.5)
axes[0].set_title('Taxa de atraso: intra vs interestadual')
axes[0].set_ylabel('Taxa de atraso')
for i, v in enumerate(cross_late.values):
    axes[0].text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')

cross_score = df.groupby('cross_state')['review_score'].mean()
axes[1].bar(['Mesmo estado', 'Interestadual'], cross_score.values,
            color=['#3498db', '#e67e22'], edgecolor='white', width=0.5)
axes[1].set_title('Nota média: intra vs interestadual')
axes[1].set_ylabel('Review score médio')
axes[1].set_ylim(3.5, 4.5)
for i, v in enumerate(cross_score.values):
    axes[1].text(i, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Conclusão H4:** Pedidos interestaduais apresentam maior taxa de atraso e menor nota média. **Hipótese confirmada.**

### Hipótese 5 — Pagamento parcelado está associado a piores avaliações

In [ ]:
df['parcelas_grupo'] = pd.cut(df['payment_installments'],
                               bins=[0,1,3,6,24], labels=['1x','2-3x','4-6x','7x+'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

parc_score = df.groupby('parcelas_grupo', observed=True)['review_score'].mean()
axes[0].bar(parc_score.index, parc_score.values,
            color=['#2ecc71','#f1c40f','#e67e22','#e74c3c'], edgecolor='white', width=0.6)
axes[0].set_title('Nota média por nº de parcelas')
axes[0].set_ylabel('Review score médio')
axes[0].set_ylim(3.5, 4.5)
for i, v in enumerate(parc_score.values):
    axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')

parc_price = df.groupby('parcelas_grupo', observed=True)['price_total'].median()
axes[1].bar(parc_price.index, parc_price.values,
            color=['#2ecc71','#f1c40f','#e67e22','#e74c3c'], edgecolor='white', width=0.6)
axes[1].set_title('Ticket mediano por nº de parcelas')
axes[1].set_ylabel('Valor mediano (R$)')
for i, v in enumerate(parc_price.values):
    axes[1].text(i, v + 1, f'R${v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Conclusão H5:** Pedidos com mais parcelas têm ticket maior e nota ligeiramente menor. **Hipótese parcialmente confirmada** — o efeito existe mas é modesto.

### Estatísticas Descritivas

In [ ]:
cols = ['review_score','delivery_delay_days','shipping_days','freight_ratio',
        'price_total','freight_total','payment_installments','n_items']
desc = df[cols].describe().round(2)
print(desc.to_string())

## 4. Modelagem

Todos os modelos foram implementados em **NumPy puro**, sem uso de frameworks de ML no treinamento. Três algoritmos foram escolhidos para cobrir diferentes abordagens:

### 4.1 Regressão Logística

Modelo linear treinado por gradient descent com regularização L2 (C=0.5, lr=0.05, 200 épocas). Para compensar o desbalanceamento de classes (79/21%), foi aplicado `class_weight='balanced'`, que pondera os erros da classe minoritária proporcionalmente à sua frequência.

**Justificativa:** modelo interpretável, eficiente computacionalmente e robusto como baseline forte. Permite analisar diretamente os coeficientes das features.

### 4.2 Naive Bayes Gaussiano

Modelo probabilístico que assume independência condicional entre features, modelando cada uma com distribuição gaussiana. A predição é feita via log-posteriors com normalização softmax para estabilidade numérica.

**Justificativa:** baseline probabilístico clássico e interpretável. Computacionalmente muito eficiente — útil para comparação de desempenho.

### 4.3 Gradient Boosting com Decision Stumps

Implementação do algoritmo de Friedman (2001) para classificação binária com log-loss. Em cada iteração, um decision stump (árvore de profundidade 1) é treinado sobre os resíduos do modelo acumulado. Otimizações: subsampling de 50% das amostras por iteração e seleção aleatória de até 12 features por stump. Parâmetros: n=100 estimadores, lr=0.15.

**Justificativa:** modelos de boosting são estado da arte em dados tabulares. A implementação com stumps é mais eficiente que árvores profundas e suficiente dado o número de features relevantes.

### 4.4 Treinamento e Validação

- **Split:** 80% treino (69.449 amostras) / 20% teste (17.362 amostras), estratificado
- **Validação cruzada:** Stratified 3-Fold no conjunto de treino
- **Métricas:** Accuracy, F1-Score (macro), Precision, Recall, ROC-AUC
- **Avaliação final:** conjunto de teste holdout (não visto durante treinamento)

## 5. Resultados e Discussão

### 5.1 Métricas no conjunto de teste

| Modelo | Accuracy | F1 (macro) | Precision | Recall | ROC-AUC |
|---|---|---|---|---|---|
| Regressão Logística | 0.7471 | **0.6290** | 0.6233 | 0.6347 | **0.6818** |
| Naive Bayes Gaussiano | 0.2240 | 0.2018 | 0.5147 | 0.3862 | 0.6469 |
| **Gradient Boosting** | **0.8292** | 0.6121 | **0.8167** | 0.5310 | 0.6633 |

### 5.2 Análise dos Resultados

O **Gradient Boosting** obteve a maior acurácia (82,9%) e precisão (81,7%), indicando que quando prediz insatisfação, acerta com alta confiabilidade. No entanto, seu recall mais baixo (53%) significa que perde parte dos casos de insatisfação.

A **Regressão Logística** apresentou o melhor F1-Score macro (0.629) e ROC-AUC (0.6818), sendo mais equilibrada entre as classes — resultado importante dado o desbalanceamento do dataset. Para um sistema de alerta de insatisfação, onde o custo de um falso negativo é alto, a Regressão Logística pode ser mais adequada.

O **Naive Bayes** teve acurácia muito baixa por conta da suposição de independência entre features, que não se sustenta neste dataset (`delivery_delay_days` e `is_late` são altamente correlacionadas). Mesmo assim, o ROC-AUC de 0.647 indica que o modelo tem poder discriminativo razoável quando as probabilidades são usadas diretamente.

### 5.3 Feature Importances — Gradient Boosting

| Feature | Importância |
|---|---|
| is_late | 33.5% |
| delivery_delay_days | 27.0% |
| shipping_days | 11.9% |
| freight_ratio | 6.8% |
| estimated_delivery_days | 4.3% |
| price_total | 3.1% |

As features relacionadas à entrega concentram mais de **72%** da importância total, confirmando as hipóteses iniciais.

In [ ]:
import json

results_path = ROOT / 'models' / 'results_final.json'
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)

    top_feat = results.get('Gradient Boosting', {}).get('top_features', {})
    if top_feat:
        feat_names = list(top_feat.keys())[:10]
        feat_vals = [top_feat[k] for k in feat_names]

        fig, ax = plt.subplots(figsize=(8, 4))
        colors_fi = ['#e74c3c' if i < 2 else '#3498db' for i in range(len(feat_names))]
        ax.barh(feat_names[::-1], feat_vals[::-1], color=colors_fi[::-1], edgecolor='white')
        ax.set_xlabel('Importância relativa')
        ax.set_title('Top 10 Features — Gradient Boosting')
        plt.tight_layout()
        plt.show()
else:
    print('results_final.json não encontrado — execute modeling.py primeiro.')

## 6. Conclusão

O projeto demonstrou que é possível prever com boa acurácia a satisfação de clientes de e-commerce utilizando dados logísticos e de pagamento disponíveis no momento da entrega. O Gradient Boosting obteve o melhor desempenho em acurácia e precisão (82,9% e 81,7%), enquanto a Regressão Logística se mostrou mais robusta ao desbalanceamento de classes, com o maior F1-Score e ROC-AUC.

A análise de importância de features confirma que **atraso na entrega** e **dias de envio** são os principais determinantes da insatisfação — resultado coerente com a literatura de satisfação em e-commerce e com as hipóteses iniciais do trabalho.

### Limitações

- O dataset cobre apenas 2016–2018 e pode não refletir padrões atuais do mercado
- O desbalanceamento 79/21% exige atenção na interpretação da acurácia
- Não foram utilizadas features textuais dos comentários dos reviews
- Os modelos foram treinados sem hiperparametrização sistemática (grid search)

### Trabalhos Futuros

- Aplicar SMOTE ou undersampling para lidar com o desbalanceamento de forma mais rigorosa
- Incorporar análise de sentimento dos textos dos reviews via TF-IDF ou BERT
- Testar modelos mais complexos como XGBoost, LightGBM e redes neurais
- Implementar monitoramento de data drift para uso em produção
- Explorar features de sazonalidade e períodos de alta demanda (Black Friday)

---

## 7. Referências

1. FRIEDMAN, J. H. Greedy Function Approximation: A Gradient Boosting Machine. *Annals of Statistics*, v. 29, n. 5, p. 1189–1232, 2001.
2. OLIST. Brazilian E-Commerce Public Dataset by Olist. Kaggle, 2018. Disponível em: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
3. HASTIE, T.; TIBSHIRANI, R.; FRIEDMAN, J. *The Elements of Statistical Learning*. 2. ed. Springer, 2009.
4. MITCHELL, T. M. *Machine Learning*. McGraw-Hill, 1997.
5. PANDAS DEVELOPMENT TEAM. pandas: powerful Python data analysis toolkit, 2023.